# CS224N L01 Word Vectors — Code Capsule: skipgram-softmax**Waypoint WP03**: Skip-gram softmax probability calculation P(o|c)**Official anchors**:- Slides p27-30 (objective function, softmax)- Notes §3.2 (Eq.4 softmax, Eq.5 cross-entropy loss)**What this capsule demonstrates**:1. Each word has TWO vectors: v_w (center) and u_w (context)2. Softmax formula: P(o|c) = exp(u_o^T v_c) / Σ_{w∈V} exp(u_w^T v_c)3. The three steps: dot product → exponentiation → normalization4. The computational bottleneck: denominator sums over ALL |V| words> This notebook uses numpy and matplotlib — both pre-installed in Google Colab.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.random.seed(42)  # reproducibility# Mini vocabulary (6 words — small enough to trace by hand)vocab = ["banking", "money", "river", "bank", "crisis", "water"]V = len(vocab)d = 4  # embedding dimension (toy example; real models use d=100-300)print(f"Vocabulary size |V| = {V}")print(f"Embedding dimension d = {d}")print(f"Words: {vocab}")

## 1. Two Vectors Per WordIn skip-gram, each word w has **two** vector representations:- **v_w** (center vector): used when w is the input/center word- **u_w** (context vector): used when w is the predicted/output wordThese are separate parameter matrices learned during training.

In [ ]:
# Center vectors V_matrix: each row is v_w for one wordV_matrix = np.array([    [ 0.5,  0.8, -0.2,  0.3],   # banking    [ 0.4,  0.7, -0.1,  0.2],   # money    [-0.3,  0.1,  0.9,  0.6],   # river    [ 0.2,  0.5,  0.4,  0.5],   # bank    [-0.5,  0.6, -0.3,  0.1],   # crisis    [-0.2,  0.0,  0.8,  0.7],   # water], dtype=np.float64)# Context vectors U_matrix: each row is u_w for one wordU_matrix = np.array([    [ 0.6,  0.7, -0.3,  0.2],   # banking    [ 0.5,  0.8, -0.2,  0.1],   # money    [-0.4,  0.2,  0.8,  0.5],   # river    [ 0.3,  0.6,  0.3,  0.4],   # bank    [-0.6,  0.5, -0.4,  0.0],   # crisis    [-0.3,  0.1,  0.7,  0.6],   # water], dtype=np.float64)print("--- Center vectors (v_w) ---")for i, w in enumerate(vocab):    print(f"  v_{w:>8s} = {V_matrix[i]}")print("\n--- Context vectors (u_w) ---")for i, w in enumerate(vocab):    print(f"  u_{w:>8s} = {U_matrix[i]}")

## 2. Softmax P(o|c) — Step by StepThe softmax formula (Notes Eq.4, Slides p28):$$P(o|c) = \frac{\exp(u_o^\top v_c)}{\sum_{w \in V} \exp(u_w^\top v_c)}$$Three steps:1. **Dot product**: compute similarity score u_w^T v_c for all w2. **Exponentiation**: exp() makes all values positive and amplifies differences3. **Normalization**: divide by Z (partition function) to get probabilities

In [ ]:
center_idx = 0  # "banking"v_c = V_matrix[center_idx]center_word = vocab[center_idx]print(f"Center word c = '{center_word}'")print(f"v_c = {v_c}")# Step 1: Dot productsprint("\n--- Step 1: Dot products (u_w^T · v_c) ---")dot_products = np.zeros(V)for i, w in enumerate(vocab):    dot_products[i] = np.dot(U_matrix[i], v_c)    print(f"  u_{w:>8s}^T · v_{center_word} = {dot_products[i]:.6f}")# Step 2: Exponentiationprint("\n--- Step 2: Exponentiation ---")exp_scores = np.exp(dot_products)for i, w in enumerate(vocab):    print(f"  exp({dot_products[i]:.6f}) = {exp_scores[i]:.6f}")# Step 3: NormalizationZ = np.sum(exp_scores)probs = exp_scores / Zprint(f"\n--- Step 3: Normalization ---")print(f"  Z (partition function) = {Z:.6f}")print(f"\n--- P(o|c='{center_word}') ---")for i, w in enumerate(vocab):    bar = "█" * int(probs[i] * 50)    print(f"  P({w:>8s}|{center_word}) = {probs[i]:.6f}  {bar}")print(f"\n  Sum = {np.sum(probs):.10f}")

## 3. Visualizing the Three Steps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))colors = ['#2196F3' if i != center_idx else '#FF5722' for i in range(V)]# Panel A: Dot productsaxes[0].barh(range(V), dot_products, color=colors, alpha=0.85)axes[0].set_yticks(range(V)); axes[0].set_yticklabels(vocab)axes[0].set_xlabel('u_w^T · v_c'); axes[0].set_title('① Dot Product')axes[0].axvline(x=0, color='gray', ls='--', alpha=0.5)# Panel B: Exponentiationaxes[1].barh(range(V), exp_scores, color=colors, alpha=0.85)axes[1].set_yticks(range(V)); axes[1].set_yticklabels(vocab)axes[1].set_xlabel('exp(u_w^T · v_c)'); axes[1].set_title('② Exponentiation')# Panel C: Normalized probabilitiesaxes[2].barh(range(V), probs, color=colors, alpha=0.85)axes[2].set_yticks(range(V)); axes[2].set_yticklabels(vocab)axes[2].set_xlabel('P(o|c="banking")'); axes[2].set_title('③ Normalize (÷Z)')axes[2].set_xlim(0, max(probs) * 1.3)plt.suptitle('Softmax Three Steps: Dot Product → Exponentiation → Normalize', fontsize=13)plt.tight_layout()plt.savefig('outputs/skipgram-softmax-three-steps.png', dpi=150, bbox_inches='tight')plt.show()

## 4. Comparison: Different Center WordsNotice how different center words produce different probability distributions over context words.

In [ ]:
test_centers = ["banking", "river", "crisis"]all_probs = {}fig, ax = plt.subplots(figsize=(10, 5))x = np.arange(V)width = 0.25for idx, cw in enumerate(test_centers):    c_idx = vocab.index(cw)    v_c = V_matrix[c_idx]    dots = U_matrix @ v_c    exp_s = np.exp(dots)    p = exp_s / np.sum(exp_s)    all_probs[cw] = p    ax.bar(x + (idx-1)*width, p, width, label=f"c='{cw}'", alpha=0.85)    print(f"\nP(o|c='{cw}'), Z={np.sum(exp_s):.4f}:")    for i, w in enumerate(vocab):        print(f"  P({w:>8s}|{cw}) = {p[i]:.6f}")ax.set_xlabel('Context word o'); ax.set_ylabel('P(o|c)')ax.set_title('Skip-gram Softmax: P(o|c) for different center words')ax.set_xticks(x); ax.set_xticklabels(vocab); ax.legend()plt.tight_layout()plt.savefig('outputs/skipgram-softmax-prob-distribution.png', dpi=150, bbox_inches='tight')plt.show()

## 5. Key Insight: Semantic SimilarityWords with similar vectors get higher probability. This is the core idea:the model learns to assign high P(o|c) to semantically related context words.

In [ ]:
p_banking = all_probs["banking"]idx_money = vocab.index("money")idx_river = vocab.index("river")print(f"When c='banking':")print(f"  P(money|banking) = {p_banking[idx_money]:.6f}  (semantically similar)")print(f"  P(river|banking) = {p_banking[idx_river]:.6f}  (semantically different)")print(f"  Ratio = {p_banking[idx_money]/p_banking[idx_river]:.2f}x")

## 6. Computational Bottleneck: Partition Function ZThe denominator Z = Σ_{w∈V} exp(u_w^T v_c) requires computing **|V| dot products** for every single training step.For a vocabulary of 50,000 words and a corpus of 10M tokens, that's 5×10¹¹ operations per epoch!This is exactly why **Negative Sampling** (WP05) replaces the full softmax with k binary classifiers.

In [ ]:
vocab_sizes = [100, 1000, 5000, 10000, 50000, 100000]T = 10_000_000ops_per_epoch = [T * vs for vs in vocab_sizes]fig, ax = plt.subplots(figsize=(9, 5))ax.semilogy(vocab_sizes, ops_per_epoch, 'bo-', linewidth=2, markersize=8)ax.set_xlabel('Vocabulary size |V|'); ax.set_ylabel('Operations per epoch (T=10M)')ax.set_title('Softmax Computational Cost vs Vocabulary Size')ax.grid(True, alpha=0.3)for vs in [10000, 50000]:    ops = T * vs    ax.annotate(f'|V|={vs:,}\n{ops:.1e} ops', (vs, ops),                textcoords="offset points", xytext=(10, 10), fontsize=9,                arrowprops=dict(arrowstyle='->', color='gray'))plt.tight_layout()plt.savefig('outputs/skipgram-softmax-computation-cost.png', dpi=150, bbox_inches='tight')plt.show()

## Summary| Concept | Formula / Value ||---|---|| Softmax probability | P(o\|c) = exp(u_o^T v_c) / Σ_w exp(u_w^T v_c) || Partition function Z | Sum over ALL \|V\| words — the bottleneck || Two vectors per word | v_w (center) and u_w (context) || Key insight | Similar vectors → higher dot product → higher P(o\|c) || Computational cost | O(\|V\|) per step → motivates Negative Sampling (WP05) |**Next**: [WP04 — Gradients & Optimization](./) and [WP05 — Negative Sampling](./)